# Vicuna 7B Medusa Optimization Benchmark

This notebook runs the final-report experiment on Google Colab:

- Plain `lmsys/vicuna-7b-v1.3` autoregressive baseline
- Official `FasterDecoding/medusa-vicuna-7b-v1.3` full Medusa tree
- Our optimized smaller-tree path, especially `turbo_fast_24`
- Optional sweep over `24,32,40,63` choices

Use an 8-bit run on T4/P100. If you get A100/L4 and enough memory, you can try the FP16 cells too.

## 1. Check GPU

In Colab: Runtime -> Change runtime type -> GPU.

In [ ]:
!nvidia-smi

import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Enable GPU in Runtime -> Change runtime type.'

## 2. Upload Or Locate Project Code

Upload a zip that contains your modified `Medusa` folder. The zip may contain either `Project/Medusa/...` or just `Medusa/...`.

Important: use the current version that includes `bench_transformers_base.py` and the modified `bench_comm_turbo.py` with `--load-in-8bit` and `--choice-sweep`.

In [ ]:
from pathlib import Path
import os, shutil, zipfile

PROJECT_ROOT = Path('/content/Project')
MEDUSA_DIR = PROJECT_ROOT / 'Medusa'

if not (MEDUSA_DIR / 'bench_comm_turbo.py').exists():
    from google.colab import files
    print('Upload a zip containing your modified Medusa folder now.')
    uploaded = files.upload()
    zip_files = [name for name in uploaded if name.lower().endswith('.zip')]
    assert zip_files, 'Please upload a .zip file containing the Medusa folder.'

    upload_root = Path('/content/uploaded_project')
    shutil.rmtree(upload_root, ignore_errors=True)
    upload_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(Path('/content') / zip_files[0]) as zf:
        zf.extractall(upload_root)

    candidates = [p for p in upload_root.rglob('Medusa') if (p / 'bench_comm_turbo.py').exists()]
    assert candidates, 'Could not find a Medusa folder with bench_comm_turbo.py in the uploaded zip.'
    source_medusa = candidates[0]

    shutil.rmtree(PROJECT_ROOT, ignore_errors=True)
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    if source_medusa.parent.name == 'Project':
        shutil.copytree(source_medusa.parent, PROJECT_ROOT, dirs_exist_ok=True)
    else:
        shutil.copytree(source_medusa, MEDUSA_DIR)

os.chdir(MEDUSA_DIR)
print('Working directory:', Path.cwd())
print('Has bench_comm_turbo:', (Path.cwd() / 'bench_comm_turbo.py').exists())
print('Has base benchmark:', (Path.cwd() / 'bench_transformers_base.py').exists())

## 3. Install Dependencies

In [ ]:
!python -m pip install -q -U \
  "transformers==4.45.2" \
  "accelerate>=0.33" \
  "tokenizers>=0.20,<0.21" \
  "safetensors>=0.4.5" \
  "bitsandbytes>=0.43.3" \
  "sentencepiece" \
  "protobuf" \
  "pandas"

!python -m pip install -q -e .

!python bench_comm_turbo.py --help | grep -E "target-new-tokens|choice-sweep|load-in-8bit|device-map"

## 3b. Colab 8-bit Medusa Head Fix

This patch keeps the custom Medusa heads in fp16 on CUDA while quantizing the Vicuna base model. It avoids a bitsandbytes device mismatch seen on Colab T4/P100.

In [ ]:
from pathlib import Path

bench_path = Path('bench_comm_turbo.py')
text = bench_path.read_text()
if 'from transformers import BitsAndBytesConfig' not in text:
    text = text.replace('import torch\n', 'import torch\nfrom transformers import BitsAndBytesConfig\n', 1)
text = text.replace('load_kwargs["load_in_8bit"] = True', 'load_kwargs["quantization_config"] = BitsAndBytesConfig(\n            load_in_8bit=True,\n            llm_int8_skip_modules=["medusa_head"],\n        )')
text = text.replace('load_kwargs["load_in_4bit"] = True', 'load_kwargs["quantization_config"] = BitsAndBytesConfig(\n            load_in_4bit=True,\n            bnb_4bit_compute_dtype=torch.float16,\n            bnb_4bit_quant_type="nf4",\n        )')
needle = '    if not (args.device_map or args.load_in_8bit or args.load_in_4bit):\n        model = model.to("cuda")\n    model = model.eval()'
replacement = '    if not (args.device_map or args.load_in_8bit or args.load_in_4bit):\n        model = model.to("cuda")\n    elif hasattr(model, "medusa_head"):\n        model.medusa_head.to(device="cuda", dtype=torch.float16)\n    model = model.eval()'
text = text.replace(needle, replacement)
bench_path.write_text(text)

base_path = Path('bench_transformers_base.py')
if base_path.exists():
    base_text = base_path.read_text()
    base_text = base_text.replace('from transformers import AutoModelForCausalLM, AutoTokenizer', 'from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig')
    base_text = base_text.replace('load_kwargs["load_in_8bit"] = True', 'load_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)')
    base_text = base_text.replace('load_kwargs["load_in_4bit"] = True', 'load_kwargs["quantization_config"] = BitsAndBytesConfig(\n            load_in_4bit=True,\n            bnb_4bit_compute_dtype=torch.float16,\n            bnb_4bit_quant_type="nf4",\n        )')
    base_path.write_text(base_text)

!python -m py_compile bench_comm_turbo.py bench_transformers_base.py
print('Applied Colab 8-bit Medusa head patch.')

## 4. Optional Hugging Face Token

Vicuna is usually accessible without a token. If Hugging Face refuses a download, add `HF_TOKEN` in Colab Secrets or paste it into `os.environ['HF_TOKEN']`.

In [ ]:
import os, subprocess

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
except Exception:
    pass

if os.environ.get('HF_TOKEN'):
    subprocess.run(['huggingface-cli', 'login', '--token', os.environ['HF_TOKEN']], check=False)
else:
    print('No HF_TOKEN set. Continuing without login.')

## 5. Benchmark Settings

For a first smoke test, use `TARGET_NEW_TOKENS = 64` and `PROMPT_SUITE = 'technical'`. For final report numbers, use `160` and `mixed`.

In [ ]:
TARGET_NEW_TOKENS = 160
MAX_STEPS = 90
PROMPT_SUITE = 'mixed'

BASE_MODEL = 'lmsys/vicuna-7b-v1.3'
MEDUSA_MODEL = 'FasterDecoding/medusa-vicuna-7b-v1.3'

print('target tokens:', TARGET_NEW_TOKENS)
print('prompt suite:', PROMPT_SUITE)

## 6. Plain Vicuna Baseline

This gives the speedup denominator for normal autoregressive decoding.

In [ ]:
!python bench_transformers_base.py \
  --model-dir {BASE_MODEL} \
  --out-csv vicuna7b_base_8bit_mixed.csv \
  --target-new-tokens {TARGET_NEW_TOKENS} \
  --prompt-suite {PROMPT_SUITE} \
  --load-in-8bit \
  --device-map auto

## 7. Main Comparison: Full Medusa Tree vs Optimized 24

This is the main final-report comparison. `medusa_base` uses the full Vicuna choice tree, while `turbo_fast_24` uses our smaller optimized tree.

In [ ]:
!python bench_comm_turbo.py \
  --model-dir {MEDUSA_MODEL} \
  --out-csv vicuna7b_full_vs_turbo_fast24_8bit_mixed.csv \
  --max-steps {MAX_STEPS} \
  --target-new-tokens {TARGET_NEW_TOKENS} \
  --prompt-suite {PROMPT_SUITE} \
  --choice-sweep 24 \
  --only medusa_base,turbo_fast_24 \
  --load-in-8bit \
  --device-map auto

## 8. Optional Sweep

Run this if time allows. It tests whether Vicuna's stronger Medusa heads benefit from a larger tree.

In [ ]:
RUN_SWEEP = True

if RUN_SWEEP:
    !python bench_comm_turbo.py \
      --model-dir {MEDUSA_MODEL} \
      --out-csv vicuna7b_choice_sweep_8bit_mixed.csv \
      --max-steps {MAX_STEPS} \
      --target-new-tokens {TARGET_NEW_TOKENS} \
      --prompt-suite {PROMPT_SUITE} \
      --choice-sweep 24,32,40,63 \
      --adaptive-sweep 24:40,32:63 \
      --load-in-8bit \
      --device-map auto
else:
    print('Sweep skipped.')

## 9. Summarize Results

In [ ]:
from pathlib import Path
import pandas as pd

base_csv = Path('vicuna7b_base_8bit_mixed.csv')
main_csv = Path('vicuna7b_full_vs_turbo_fast24_8bit_mixed.csv')
sweep_csv = Path('vicuna7b_choice_sweep_8bit_mixed.csv')

base = pd.read_csv(base_csv)
base_tps = base['tps'].mean()
print(f'Plain Vicuna base mean TPS: {base_tps:.2f}')

def summarize(path):
    if not path.exists():
        print('Missing', path)
        return None
    df = pd.read_csv(path)
    grouped = df.groupby('mode', as_index=False).agg(
        tps=('tps', 'mean'),
        accepted_tokens_per_step=('accepted_tokens_per_step', 'mean'),
        verified_nodes_per_step=('verified_nodes_per_step', 'mean'),
        prefix_match_vs_base=('prefix_match_vs_base', 'mean'),
    )
    grouped['speedup_vs_plain_base'] = grouped['tps'] / base_tps
    medusa_base_tps = grouped.loc[grouped['mode'].eq('medusa_base'), 'tps']
    if len(medusa_base_tps):
        grouped['speedup_vs_medusa_base'] = grouped['tps'] / float(medusa_base_tps.iloc[0])
    grouped = grouped.sort_values('tps', ascending=False)
    print('\n' + str(path))
    display(grouped)
    return grouped

main_summary = summarize(main_csv)
sweep_summary = summarize(sweep_csv)

print('\nCSV files:')
!ls -lh vicuna7b*_8bit_mixed.csv

## 10. Download CSVs

In [ ]:
from google.colab import files
for name in ['vicuna7b_base_8bit_mixed.csv', 'vicuna7b_full_vs_turbo_fast24_8bit_mixed.csv', 'vicuna7b_choice_sweep_8bit_mixed.csv']:
    if Path(name).exists():
        files.download(name)